# ♟️ AI Chess — Training on Lichess dataset
Trains `ChessNet` on `lichess.pgn.zst` from Google Drive.
- **50 chunks × 20 000 games × 10 epochs** per chunk
- Model is saved to Google Drive after every chunk
- Resumes automatically if `model.pt` already exists

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Paths — edit if your Drive layout differs

In [2]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/aichess"

PGN_FILE = os.path.join(DRIVE_ROOT, "lichess.pgn.zst")
WEIGHTS_DIR = os.path.join(DRIVE_ROOT, "weights")
SAVE_PATH = os.path.join(WEIGHTS_DIR, "model.pt")

os.makedirs(WEIGHTS_DIR, exist_ok=True)

print("PGN  :", PGN_FILE, " exists:", os.path.exists(PGN_FILE))
print("Model:", SAVE_PATH, " exists:", os.path.exists(SAVE_PATH))

PGN  : /content/drive/MyDrive/aichess/lichess.pgn.zst  exists: True
Model: /content/drive/MyDrive/aichess/weights/model.pt  exists: True


## 3. Clone / pull the repo & install dependencies

In [3]:
REPO_DIR = "/content/ai_chess"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone https://github.com/SezamParmezan/ai_chess {REPO_DIR}
    %cd {REPO_DIR}

!pip install -q python-chess torch

Cloning into '/content/ai_chess'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 110 (delta 40), reused 98 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (110/110), 39.75 MiB | 23.35 MiB/s, done.
Resolving deltas: 100% (40/40), done.
/content/ai_chess
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 75.4 MB/s eta 0:00:00:00:010:01
  Preparing metadata (setup.py) ... done


## 4. Add project paths to sys.path

In [4]:
import sys

for p in [REPO_DIR, os.path.join(REPO_DIR, "app", "ml")]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("sys.path:", sys.path[:4])

sys.path: ['/content/ai_chess/app/ml', '/content/ai_chess', '/content', '/env/python']


## 5. Verify GPU

In [5]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## 6. Import project modules

In [6]:
from encode import encode_game
from dataset import load_games
from model import ChessNet, save_model, load_model
from train import ChessDataset

print("All modules imported successfully")

All modules imported successfully


## 7. Training — C chunks × G games × E epochs

In [ ]:
import torch
import torch.nn as nn
import os
import time
import random
from collections import deque
from torch.utils.data import DataLoader

# ── Hyperparameters ──────────────────────────────────────────────────
CHUNKS           = 50
GAMES_PER_CHUNK  = 10_000
EPOCHS_PER_CHUNK = 8
START_CHUNK      = 4
BATCH_SIZE       = 512
LR               = 1e-3
REPLAY_SIZE      = 300_000 
# ────────────────────────────────────────────────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

replay_buffer = deque(maxlen=REPLAY_SIZE)

# Resume or start fresh
if os.path.exists(SAVE_PATH):
    print(f"Resuming from {SAVE_PATH}")
    model = load_model(SAVE_PATH)
else:
    print("Starting fresh model")
    model = ChessNet()

model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
policy_loss_fn = nn.CrossEntropyLoss()
value_loss_fn = nn.MSELoss()

# ── Main loop ────────────────────────────────────────────────────────
for chunk_idx in range(max(0, START_CHUNK - 1), CHUNKS):
    t0     = time.time()
    offset = chunk_idx * GAMES_PER_CHUNK

    print(f"\n{'='*60}")
    print(f"CHUNK {chunk_idx+1}/{CHUNKS}  |  games {offset} – {offset+GAMES_PER_CHUNK}")
    print(f"{'='*60}")

    games = load_games(PGN_FILE, max_games=GAMES_PER_CHUNK, skip=offset)
    if not games:
        print("No more games — training complete.")
        break

    # Encode
    samples = []
    for i, game in enumerate(games):
        samples.extend(encode_game(game))
        if i % 5000 == 0:
            print(f" Encoded {i}/{len(games)} games  ({len(samples)} samples so far)")

    print(f" Total samples in chunk: {len(samples)}")

    # Replay buffer
    for s in samples:
        replay_buffer.append(s)

    # Build training set from replay buffer
    train_samples = list(replay_buffer)
    random.shuffle(train_samples)

    dataset = ChessDataset(train_samples)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE,
                         shuffle=True, num_workers=2, pin_memory=True)

    # Train epochs
    for epoch in range(EPOCHS_PER_CHUNK):
        model.train()
        total_p = total_v = 0.0

        for st, act, val in loader:
            st = st.to(device)
            act = act.to(device)
            val = val.to(device).unsqueeze(1)

            pred_policy, pred_value = model(st)

            p_loss = policy_loss_fn(pred_policy, act)
            v_loss = value_loss_fn(pred_value, val)
            loss = p_loss + v_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_p += p_loss.item()
            total_v += v_loss.item()

        b = len(loader)
        print(f"  Epoch {epoch+1:>2}/{EPOCHS_PER_CHUNK} | "
              f"policy: {total_p/b:.4f} | value: {total_v/b:.4f}")

    save_model(model, SAVE_PATH)
    elapsed = time.time() - t0
    print(f"Saved at {SAVE_PATH}  (chunk took {elapsed/60:.1f} min)")

print("\nAll done!")

Device: cuda
Resuming from /content/drive/MyDrive/aichess/weights/model.pt

CHUNK 4/50  |  games 30000 – 40000
Skipping 30000 games...
 Encoded 0/10000 games  (64 samples so far)
 Encoded 5000/10000 games  (338133 samples so far)
 Total samples in chunk: 672409
  Epoch  1/8 | policy: 2.3509 | value: 0.7974
  Epoch  2/8 | policy: 1.9109 | value: 0.5909
  Epoch  3/8 | policy: 1.5928 | value: 0.4190
  Epoch  4/8 | policy: 1.2816 | value: 0.3189
  Epoch  5/8 | policy: 0.9879 | value: 0.2620
  Epoch  6/8 | policy: 0.7468 | value: 0.2299
  Epoch  7/8 | policy: 0.5774 | value: 0.2098
  Epoch  8/8 | policy: 0.4714 | value: 0.1950
Saved at /content/drive/MyDrive/aichess/weights/model.pt  (chunk took 11.9 min)

CHUNK 5/50  |  games 40000 – 50000
Skipping 40000 games...
 Encoded 0/10000 games  (98 samples so far)
 Encoded 5000/10000 games  (336568 samples so far)
 Total samples in chunk: 668721
  Epoch  1/8 | policy: 2.3865 | value: 0.8221
  Epoch  2/8 | policy: 1.9336 | value: 0.6620
  Epoch  3/